# CODA v0.1 — Coupled Ocean Diffusion for task-aware Chl-a reconstruction + probabilistic PFZ (South China Sea)

**Status: initial research code, not yet validated on real data.** Treat every default below as a starting point to be justified in the paper, not as a result.

**What this notebook does**
1. Builds a daily 0.1° SCS data cube: MODIS-Aqua Chl-a (ERDDAP, no login), CMEMS OSTIA SST + DUACS SLA/geostrophic currents (free CMEMS account), GFW apparent fishing effort (free API token or uploaded CSVs).
2. Builds leakage-safe chronological splits with 15-day buffers, training-only normalisation, a training-only PFZ threshold, and fixed evaluation masks (random / real-cloud transfer / persistent-monsoon) shared by every method.
3. Trains **CODA**: a pixel-space conditional diffusion model (v-prediction) whose one-step clean estimate feeds a PFZ head. In `coupled` mode PFZ gradients flow into the denoiser; in `detached` mode (the uncoupled / sequential baseline) they are stopped. Same architecture, same data, same seed → a clean test of the central hypothesis.
4. Baselines: DINEOF (numpy re-implementation), Gaussian U-Net (DINCAE-style, same backbone), climatology fill, env-only PFZ, climatology PFZ, persistence PFZ.
5. Evaluates RMSE/MAE/CRPS/spread-skill/90%-coverage, front F1 + gradient error, masked SSIM, radially averaged PSD, and PFZ AUROC/AUPRC/Brier/BSS-vs-climatology/ECE/IoU, with paired Wilcoxon + block-bootstrap CIs.

**How to run on Kaggle (free tier)** — set `SESSION` in the config cell
- `"smoke"` (GPU on, Internet not needed): whole pipeline on synthetic data with tiny settings. Run this first; it tells you whether anything is broken before you spend GPU quota.
- `"data"` (CPU session is enough, Internet ON): downloads, preprocesses, builds the fixed evaluation masks, runs DINEOF + climatology/persistence baselines. Then *Save Version* and publish `/kaggle/working/processed` (and `runs/`) as a Kaggle Dataset, e.g. `coda-processed`.
- `"train"` (GPU): set `PROCESSED_IN` to that dataset and put one or two `(mode, seed)` pairs in `RUN_PLAN`, e.g. `("coupled",0)`. Next session `("detached",0)`, then seeds 1, 2, ... Training stops before `MAX_TRAIN_MINUTES`; attach the session output as input next time and it resumes from `ckpt.pt`.
- `"gpu_baselines"` (GPU): Gaussian U-Net (DINCAE-style) and env-only PFZ.
- `"compare"` (CPU): attach every session's output; builds the tables, paired tests and figures.

**Secrets (Add-ons → Secrets):** `CMEMS_USER`, `CMEMS_PASS`, `GFW_TOKEN`.

**Things I could not verify from here and you must check:** exact CMEMS dataset IDs/variable names at run time, the GFW API JSON layout for daily reports, Kaggle session/GPU limits, and all runtime estimates. The code prints what it finds so mismatches surface early.

In [ ]:
# ============================== CONFIG ==============================
import os, sys, json, math, time, glob, copy, random, warnings, contextlib
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- choose what THIS Kaggle session does ----
SESSION = "smoke"            # "smoke" | "data" | "train" | "gpu_baselines" | "compare"
PROCESSED_IN = None          # e.g. "/kaggle/input/coda-processed"  (needed for train / gpu_baselines / compare)
RUN_PLAN = [("coupled", 0)]  # (PFZ_MODE, seed) for "train": one or two entries per GPU session
_ALL_OFF = dict(download=False, preprocess=False, baselines_cpu=False, dineof=False, train=False,
                baseline_unet=False, baseline_env_pfz=False, compare=False)
PRESETS = {
    "smoke": dict(DATA_MODE="synthetic", RUN_PLAN=[("coupled", 0), ("detached", 0)],
                  STAGES={**{k: True for k in _ALL_OFF}, "download": False}),
    "data": dict(DATA_MODE="real", STAGES={**_ALL_OFF, "download": True, "preprocess": True,
                                           "baselines_cpu": True, "dineof": True}),
    "train": dict(DATA_MODE="real", STAGES={**_ALL_OFF, "train": True}),
    "gpu_baselines": dict(DATA_MODE="real", STAGES={**_ALL_OFF, "baseline_unet": True, "baseline_env_pfz": True}),
    "compare": dict(DATA_MODE="real", STAGES={**_ALL_OFF, "compare": True}),
}

CFG = dict(
    DATA_MODE=PRESETS[SESSION]["DATA_MODE"],
    RUN_PLAN=PRESETS[SESSION].get("RUN_PLAN", RUN_PLAN),
    STAGES=PRESETS[SESSION]["STAGES"],
    PROCESSED_IN=PROCESSED_IN,

    EXTRA_RUN_GLOB="/kaggle/input/**/runs/*",   # earlier sessions' run folders, for resume + final comparison
    # ---- domain (TO BE FINALIZED with co-authors) ----
    BBOX=dict(lon_min=105.0, lon_max=121.0, lat_min=5.0, lat_max=23.0),
    GRID_RES=0.1,                     # deg; matches GFW 'LOW' (0.1 deg) cells exactly
    DATE_START="2015-01-01", DATE_END="2023-12-31",
    SPLIT=dict(train=("2015-01-01", "2020-12-31"),
               val=("2021-01-16", "2021-12-31"),     # 15-day buffers against autocorrelation leakage
               test=("2022-01-16", "2023-12-31")),
    # ---- sources (verify IDs at run time) ----
    ERDDAP_BASE="https://coastwatch.pfeg.noaa.gov/erddap/griddap",
    CHL_DATASET="erdMH1chla1day_R2022SQ", CHL_VAR="chlor_a",
    SST_DATASET="METOFFICE-GLO-SST-L4-REP-OBS-SST", SST_VAR="analysed_sst",
    SLA_DATASETS=["cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D",
                  "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.25deg_P1D"],
    SLA_VARS=["sla", "ugos", "vgos"],
    GFW_MODE="api",                   # "api" | "csv" (CSVs uploaded as a Kaggle dataset)
    GFW_DATASET="public-global-fishing-effort:latest",
    GFW_CSV_GLOB="/kaggle/input/**/*.csv",
    GFW_CSV_COLS=dict(date="date", lat="cell_ll_lat", lon="cell_ll_lon", hours="fishing_hours"),
    # ---- preprocessing ----
    CHL_MIN=0.01, CHL_MAX=100.0, MIN_SUBPIX_VALID=3,
    # ---- PFZ proxy (TO BE FINALIZED; fixed from TRAIN data only) ----
    PFZ_WINDOW=(-3, 5),               # label at t = effort summed over days [t-3, t+4] (8 days)
    PFZ_QUANTILE=0.75,                # PFZ = window effort >= 75th pct of NONZERO train effort
    PFZ_FOOTPRINT_MIN_DAYS=5,         # cells with effort on >= N train days form the evaluation footprint
    PFZ_POS_WEIGHT=1.0,               # keep 1.0 for calibrated probabilities
    # ---- masks ----
    CONTEXT_OFFSETS=[-1, 1],          # neighbour days given as (masked) context; use [-2,-1] for a past-only setup
    TRAIN_MASK_PROBS=dict(random=0.2, cloud=0.45, blob=0.1, persistent=0.25),
    RANDOM_DROP=(0.3, 0.9), EVAL_RANDOM_DROP=0.7, PERSIST_DAYS=3, N_MONSOON_MONTHS=3,
    N_EVAL_DAYS=40, N_VAL_EVAL_DAYS=12, EVAL_MIN_VALID_FRAC=0.15, EVAL_SEED=2024,
    # ---- model / training ----
    PATCH=64, BATCH=32, BASE_CH=64, CH_MULT=(1, 2, 2, 2), N_RES=2, ATTN_RES=(8,), DROPOUT=0.1,
    PFZ_BASE_CH=32, T_DIFF=1000, LR=2e-4, WARMUP=1000, EMA=0.999, AMP=True, WORKERS=3,
    MAX_STEPS=40000, MAX_TRAIN_MINUTES=360, LOG_EVERY=200, CKPT_EVERY=2000,
    LAMBDA=dict(diff=1.0, recon=0.5, grad=0.5, pfz=0.5, adv=0.05),
    PFZ_TMAX_FRAC=0.3, LOWT_FRAC=0.5, W_OBSERVED=0.25, MIN_PATCH_OCEAN=0.5, MIN_PATCH_VALID=0.25,
    UNET_STEPS=20000, ENV_PFZ_STEPS=8000,
    # ---- sampling / evaluation ----
    DDIM_STEPS=25, DDIM_ETA=1.0, N_SAMPLES=16, N_SAMPLES_UQ=50, N_UQ_DAYS=10, SAMPLE_CHUNK=16,
    TILE_STRIDE=48, PSD_WIN=32, KEEP_MAPS=3,
    DINEOF_KMAX=30, DINEOF_MAXITER=40,
    WORK="/kaggle/working" if os.path.isdir("/kaggle/working") else "./work",
)

if CFG["DATA_MODE"] == "synthetic":   # small, fast settings for the smoke test
    CFG.update(DATE_START="2015-01-01", DATE_END="2016-12-31",
               SPLIT=dict(train=("2015-01-01", "2016-03-31"), val=("2016-04-16", "2016-07-31"),
                          test=("2016-08-16", "2016-12-31")),
               MAX_STEPS=600, BASE_CH=32, BATCH=16, UNET_STEPS=400, ENV_PFZ_STEPS=300, WARMUP=100,
               LOG_EVERY=50, CKPT_EVERY=300, N_EVAL_DAYS=6, N_VAL_EVAL_DAYS=3, N_SAMPLES=6,
               N_SAMPLES_UQ=12, N_UQ_DAYS=2, DDIM_STEPS=8, DINEOF_KMAX=8, DINEOF_MAXITER=15,
               MAX_TRAIN_MINUTES=20)
print("SESSION:", SESSION); print(json.dumps({k: CFG[k] for k in ["DATA_MODE", "RUN_PLAN", "STAGES", "BBOX", "GRID_RES", "SPLIT"]}, indent=1, default=str))

In [ ]:
# ============================== INSTALLS (real mode only) ==============================
import subprocess
if CFG["DATA_MODE"] == "real" and CFG["STAGES"]["download"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "copernicusmarine", "netCDF4", "h5netcdf"], check=False)

In [ ]:
# ============================== HELPERS ==============================
WORK = Path(CFG["WORK"]); RAW = WORK / "raw"; PROC = WORK / "processed"; RUNS = WORK / "runs"; FIGS = WORK / "figs"
for p in (RAW, PROC, RUNS, FIGS):
    p.mkdir(parents=True, exist_ok=True)
ENV_VARS = ["sst", "sla", "ugos", "vgos"]
KIND_ID = dict(random=1, cloud=2, blob=3, persistent=4)
EVAL_KINDS = ["random", "cloud", "persistent"]

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

def make_grid(cfg):
    b, r = cfg["BBOX"], cfg["GRID_RES"]
    ny = int(round((b["lat_max"] - b["lat_min"]) / r)); nx = int(round((b["lon_max"] - b["lon_min"]) / r))
    lat = b["lat_min"] + r * (np.arange(ny) + 0.5)      # ascending: row 0 = south
    lon = b["lon_min"] + r * (np.arange(nx) + 0.5)
    return lat, lon

def cell_index(vals, start, res, n):
    idx = np.floor((np.asarray(vals, np.float64) - start) / res + 1e-6).astype(np.int64)
    idx[(idx < 0) | (idx >= n)] = -1
    return idx

def tile_positions(n, tile, stride):
    if n <= tile:
        return [0]
    p = list(range(0, n - tile + 1, stride))
    if p[-1] != n - tile:
        p.append(n - tile)
    return p

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

def sobel_np(x):
    from scipy.ndimage import sobel
    return sobel(x, axis=1) / 8.0, sobel(x, axis=0) / 8.0

def valid3(v):
    from scipy.ndimage import minimum_filter
    return minimum_filter(v.astype(np.uint8), size=3, mode="constant", cval=0).astype(bool)

def geostrophic_uv(eta, lat, res):
    g, om = 9.81, 7.2921e-5
    f = 2 * om * np.sin(np.deg2rad(lat))[:, None]
    f = np.where(np.abs(f) < 1e-5, 1e-5, f)
    dy = res * 110540.0; dx = res * 111320.0 * np.cos(np.deg2rad(lat))[:, None]
    return -g / f * np.gradient(eta, axis=0) / dy, g / f * np.gradient(eta, axis=1) / dx

In [ ]:
# ============================== SYNTHETIC DATA (smoke test only) ==============================
def smooth_noise(rng, shape, sigma):
    from scipy.ndimage import gaussian_filter
    a = gaussian_filter(rng.standard_normal(shape), sigma, mode="wrap")
    return (a - a.mean()) / (a.std() + 1e-8)

def make_synthetic(cfg, seed=0):
    """Fake but structured fields (eddies, a meandering Chl front, coastal upwelling, AR(1) clouds,
    effort tied to fronts). Only for checking that the code runs; it says nothing about real skill."""
    rng = np.random.default_rng(seed)
    lat, lon = make_grid(cfg); H, W = len(lat), len(lon)
    dates = pd.date_range(cfg["DATE_START"], cfg["DATE_END"], freq="D"); T = len(dates)
    yy, xx = np.meshgrid(np.linspace(0, 1, H), np.linspace(0, 1, W), indexing="ij")
    land = (xx < 0.06 + 0.04 * np.sin(7 * yy)) | (((yy - 0.72) ** 2 + (xx - 0.78) ** 2) < 0.003)
    K = 14
    ey, ex = rng.uniform(0.05, 0.95, K), rng.uniform(0.1, 1.0, K)
    amp = rng.choice([-1.0, 1.0], K) * rng.uniform(0.05, 0.15, K)
    rad, spd = rng.uniform(0.04, 0.08, K), rng.uniform(0.001, 0.003, K)
    doy = dates.dayofyear.values
    out = {k: np.zeros((T, H, W), np.float32) for k in ["logchl", "sst", "sla", "ugos", "vgos", "effort"]}
    cloud = smooth_noise(rng, (H, W), 6)
    footprint = (xx > 0.12) & (xx < 0.85) & ~land
    for t in range(T):
        season = np.cos(2 * np.pi * (doy[t] - 20) / 365.25)
        exx = (ex - spd * t) % 1.0
        sla = 0.02 * smooth_noise(rng, (H, W), 10)
        for k in range(K):
            dxk = np.minimum(np.abs(xx - exx[k]), 1 - np.abs(xx - exx[k]))
            sla = sla + amp[k] * np.exp(-((yy - ey[k]) ** 2 + dxk ** 2) / rad[k] ** 2)
        u, v = geostrophic_uv(sla, lat, cfg["GRID_RES"])
        upw = np.exp(-((xx - 0.12) / 0.05) ** 2) * max(0.0, -season)
        sst = 29 - 3.5 * yy * (0.6 + 0.4 * season) - 2.0 * upw + 8.0 * sla + 0.2 * smooth_noise(rng, (H, W), 4)
        yf = 0.45 + 0.06 * np.sin(9 * xx + 0.03 * t)
        logchl = (-0.9 + 0.9 * np.exp(-np.clip(xx - 0.06, 0, None) / 0.08) + 0.8 * upw - 0.15 * (sst - 27)
                  - 3.0 * sla + 0.25 * np.tanh((yy - yf) / 0.015) + 0.08 * smooth_noise(rng, (H, W), 2))
        cloud = 0.75 * cloud + np.sqrt(1 - 0.75 ** 2) * smooth_noise(rng, (H, W), 6)
        cfrac = np.clip(0.55 + 0.25 * np.sin(2 * np.pi * (doy[t] - 120) / 365.25), 0.1, 0.95)
        clear = (cloud > np.quantile(cloud, cfrac)) & ~land
        gy_, gx_ = np.gradient(logchl)
        lam = footprint * 1.5 / (1 + np.exp(-(2.0 * np.hypot(gx_, gy_) / 0.02 + 1.5 * (logchl + 0.6) - 2.5)))
        eff = rng.poisson(lam) * rng.gamma(2.0, 2.0, (H, W))
        out["logchl"][t] = np.where(clear, logchl, np.nan)
        for k, a in (("sst", sst), ("sla", sla), ("ugos", u), ("vgos", v)):
            out[k][t] = np.where(land, np.nan, a)
        out["effort"][t] = np.where(land, 0.0, eff)
    out.update(lat=lat, lon=lon, dates=dates.values.astype("datetime64[D]"), land=land)
    return out

In [ ]:
# ============================== DOWNLOADS (real mode) ==============================
def month_ranges(start, end):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    for m0 in pd.date_range(s.replace(day=1), e, freq="MS"):
        yield max(m0, s), min(m0 + pd.offsets.MonthEnd(0), e)

def http_get(url, out, tries=4, timeout=900):
    import requests
    for k in range(tries):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                if 400 <= r.status_code < 500:
                    log(f"  HTTP {r.status_code} (not retrying): {r.text[:200]}"); return False
                r.raise_for_status()
                tmp = str(out) + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
                os.replace(tmp, out); return True
        except Exception as e:
            log(f"  retry {k + 1}/{tries}: {e}"); time.sleep(15 * (k + 1))
    return False

def download_chl(cfg):
    d = RAW / "chl"; d.mkdir(exist_ok=True); b = cfg["BBOX"]; pad = 0.1
    for t0, t1 in month_ranges(cfg["DATE_START"], cfg["DATE_END"]):
        out = d / f"chl_{t0:%Y_%m}.nc"
        if out.exists() and out.stat().st_size > 1000:
            continue
        tq = f"[({t0:%Y-%m-%d}T12:00:00Z):1:({t1:%Y-%m-%d}T12:00:00Z)]"
        lonq = f"[({b['lon_min'] - pad}):1:({b['lon_max'] + pad})]"
        ok = False
        for latq in (f"[({b['lat_max'] + pad}):1:({b['lat_min'] - pad})]",      # dataset latitude is descending
                     f"[({b['lat_min'] - pad}):1:({b['lat_max'] + pad})]"):
            url = f"{cfg['ERDDAP_BASE']}/{cfg['CHL_DATASET']}.nc?{cfg['CHL_VAR']}{tq}{latq}{lonq}"
            if http_get(url, out):
                ok = True; break
        log(("ok " if ok else "FAILED ") + out.name)

def download_cmems(cfg):
    import copernicusmarine
    user, pwd = get_secret("CMEMS_USER"), get_secret("CMEMS_PASS")
    assert user and pwd, "Add CMEMS_USER / CMEMS_PASS as Kaggle secrets"
    b = cfg["BBOX"]; pad = 0.3
    common = dict(minimum_longitude=b["lon_min"] - pad, maximum_longitude=b["lon_max"] + pad,
                  minimum_latitude=b["lat_min"] - pad, maximum_latitude=b["lat_max"] + pad,
                  username=user, password=pwd, output_directory=str(RAW))
    for y in range(pd.Timestamp(cfg["DATE_START"]).year, pd.Timestamp(cfg["DATE_END"]).year + 1):
        s, e = f"{y}-01-01T00:00:00", f"{y}-12-31T23:59:59"
        if not (RAW / f"sst_{y}.nc").exists():
            copernicusmarine.subset(dataset_id=cfg["SST_DATASET"], variables=[cfg["SST_VAR"]], start_datetime=s,
                                    end_datetime=e, output_filename=f"sst_{y}.nc", **common)
        if not (RAW / f"sla_{y}.nc").exists():
            for ds_id in cfg["SLA_DATASETS"]:
                try:
                    copernicusmarine.subset(dataset_id=ds_id, variables=cfg["SLA_VARS"], start_datetime=s,
                                            end_datetime=e, output_filename=f"sla_{y}.nc", **common)
                    log("SLA from", ds_id); break
                except Exception as ex:
                    log("SLA dataset failed:", ds_id, str(ex)[:200])
        log("CMEMS", y, "done")

def _gfw_flatten(o):
    rows, stack = [], [o]
    while stack:
        x = stack.pop()
        if isinstance(x, dict):
            if "lat" in x and "lon" in x and "date" in x and ("hours" in x or "value" in x):
                rows.append(x)
            else:
                stack.extend(x.values())
        elif isinstance(x, list):
            stack.extend(x)
    return rows

def download_gfw_api(cfg):
    """4Wings report, apparent fishing effort, 0.1 deg (LOW), DAILY. The request layout follows the public docs;
    the JSON layout of DAILY reports is parsed defensively - inspect gfw_debug.json if you get 0 rows."""
    import requests
    token = get_secret("GFW_TOKEN"); assert token, "Add GFW_TOKEN as a Kaggle secret"
    d = RAW / "gfw"; d.mkdir(exist_ok=True); b = cfg["BBOX"]
    ring = [[b["lon_min"], b["lat_min"]], [b["lon_max"], b["lat_min"]], [b["lon_max"], b["lat_max"]],
            [b["lon_min"], b["lat_max"]], [b["lon_min"], b["lat_min"]]]
    fc = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {},
                                                     "geometry": {"type": "Polygon", "coordinates": [ring]}}]}
    url = "https://gateway.api.globalfishingwatch.org/v3/4wings/report"
    hdr = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    for t0, t1 in month_ranges(cfg["DATE_START"], cfg["DATE_END"]):
        out = d / f"gfw_{t0:%Y_%m}.csv"
        if out.exists():
            continue
        params = {"spatial-resolution": "LOW", "temporal-resolution": "DAILY", "spatial-aggregation": "false",
                  "datasets[0]": cfg["GFW_DATASET"], "format": "JSON",
                  "date-range": f"{t0:%Y-%m-%d},{t1 + pd.Timedelta(days=1):%Y-%m-%d}"}
        rows = None
        for body in ({"geojson": fc}, {"geojson": json.dumps(fc)}):
            for k in range(3):
                try:
                    r = requests.post(url, params=params, headers=hdr, json=body, timeout=900)
                    if r.status_code == 429:
                        time.sleep(60); continue
                    if r.status_code >= 400:
                        log("GFW HTTP", r.status_code, r.text[:300]); break
                    js = r.json()
                    if not (d / "gfw_debug.json").exists():
                        (d / "gfw_debug.json").write_text(json.dumps(js)[:20000])
                    rows = _gfw_flatten(js); break
                except Exception as e:
                    log("GFW error", e); time.sleep(20)
            if rows is not None:
                break
        if rows is None:
            log("FAILED", out.name); continue
        df = pd.DataFrame(rows)
        if len(df) == 0:
            log("WARNING 0 rows for", out.name, "- inspect raw/gfw/gfw_debug.json")
            df = pd.DataFrame(columns=["date", "lat", "lon", "hours"])
        else:
            df = df.rename(columns={"value": "hours"})[["date", "lat", "lon", "hours"]]
            df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
            df = df[(df.date >= f"{t0:%Y-%m-%d}") & (df.date <= f"{t1:%Y-%m-%d}")]
        df.to_csv(out, index=False); log("GFW", out.name, len(df), "rows"); time.sleep(2)

In [ ]:
# ============================== PREPROCESS (real mode) ==============================
def _coord(ds, names):
    for n in names:
        if n in ds.coords or n in ds.dims:
            return n
    raise KeyError(f"none of {names} in {list(ds.coords)}")

def preprocess_real(cfg):
    import xarray as xr
    lat, lon = make_grid(cfg); H, W = len(lat), len(lon); r = cfg["GRID_RES"]
    dates = pd.date_range(cfg["DATE_START"], cfg["DATE_END"], freq="D"); T = len(dates)
    d0 = np.datetime64(dates[0].date(), "D")
    lat0, lon0 = lat[0] - r / 2, lon[0] - r / 2

    def tpos(tv):
        k = (np.asarray(tv).astype("datetime64[D]") - d0).astype(np.int64)
        return np.where((k >= 0) & (k < T), k, -1)

    def bin_files(files, var, transform=None, min_count=1):
        A = np.full((T, H, W), np.nan, np.float32)
        for f in files:
            ds = xr.open_dataset(f)
            la, lo = _coord(ds, ["latitude", "lat"]), _coord(ds, ["longitude", "lon"])
            LA, LO = np.meshgrid(ds[la].values, ds[lo].values, indexing="ij")
            iy, ix = cell_index(LA, lat0, r, H), cell_index(LO, lon0, r, W)
            inside = (iy >= 0) & (ix >= 0); flat = (iy * W + ix)[inside]
            arr = ds[var].values
            for k, ti in enumerate(tpos(ds["time"].values)):
                if ti < 0:
                    continue
                a = arr[k][inside].astype(np.float64)
                if transform is not None:
                    a = transform(a)
                good = np.isfinite(a)
                s = np.bincount(flat[good], weights=a[good], minlength=H * W)
                c = np.bincount(flat[good], minlength=H * W)
                m = np.full(H * W, np.nan, np.float32); ok = c >= min_count; m[ok] = s[ok] / c[ok]
                A[ti] = m.reshape(H, W)
            ds.close()
        return A

    def interp_files(files, var):
        A = np.full((T, H, W), np.nan, np.float32)
        for f in files:
            ds = xr.open_dataset(f)
            la, lo = _coord(ds, ["latitude", "lat"]), _coord(ds, ["longitude", "lon"])
            vals = ds[var].interp({la: lat, lo: lon}, method="linear").values
            for k, ti in enumerate(tpos(ds["time"].values)):
                if ti >= 0:
                    A[ti] = vals[k]
            ds.close()
        return A

    D = dict(lat=lat, lon=lon, dates=dates.values.astype("datetime64[D]"))
    log("binning Chl (geometric mean of 4 km pixels) ...")
    D["logchl"] = bin_files(sorted((RAW / "chl").glob("chl_*.nc")), cfg["CHL_VAR"],
                            transform=lambda a: np.where(a > 0, np.log10(np.clip(a, cfg["CHL_MIN"], cfg["CHL_MAX"])), np.nan),
                            min_count=cfg["MIN_SUBPIX_VALID"])
    log("binning SST ...")
    sst = bin_files(sorted(RAW.glob("sst_*.nc")), cfg["SST_VAR"])
    if np.nanmean(sst) > 200:
        sst -= 273.15
    D["sst"] = sst
    log("interpolating SLA / currents ...")
    for v in cfg["SLA_VARS"]:
        D[v] = interp_files(sorted(RAW.glob("sla_*.nc")), v)
    D["land"] = ~(np.isfinite(D["sst"]).mean(0) > 0.95)
    log("gridding GFW effort ...")
    eff = np.zeros((T, H, W), np.float32)
    if cfg["GFW_MODE"] == "api":
        files, cols = sorted((RAW / "gfw").glob("gfw_*.csv")), dict(date="date", lat="lat", lon="lon", hours="hours")
    else:
        files, cols = sorted(glob.glob(cfg["GFW_CSV_GLOB"], recursive=True)), cfg["GFW_CSV_COLS"]
    for f in files:
        for ch in pd.read_csv(f, usecols=list(cols.values()), chunksize=2_000_000):
            ti = tpos(pd.to_datetime(ch[cols["date"]]).values)
            iy, ix = cell_index(ch[cols["lat"]].values, lat0, r, H), cell_index(ch[cols["lon"]].values, lon0, r, W)
            ok = (ti >= 0) & (iy >= 0) & (ix >= 0)
            np.add.at(eff, (ti[ok], iy[ok], ix[ok]), ch[cols["hours"]].values[ok].astype(np.float32))
    eff[:, D["land"]] = 0
    D["effort"] = eff
    log("done. Chl valid fraction over ocean:", float(np.isfinite(D["logchl"][:, ~D["land"]]).mean()))
    return D

def save_processed(D, d):
    d = Path(d); d.mkdir(parents=True, exist_ok=True)
    for k in ["logchl", "sst", "sla", "ugos", "vgos", "effort"]:
        np.save(d / f"{k}.npy", D[k].astype(np.float32))
    for k in ["land", "lat", "lon", "dates"]:
        np.save(d / f"{k}.npy", D[k])

def load_processed(d):
    d = Path(d)
    return {k: np.load(d / f"{k}.npy", allow_pickle=False)
            for k in ["logchl", "sst", "sla", "ugos", "vgos", "effort", "land", "lat", "lon", "dates"]}

In [ ]:
# ============================== PREPARE: splits, normalisation, PFZ labels (TRAIN-only decisions) ==============================
def prepare(D, cfg):
    dates = pd.DatetimeIndex(D["dates"]); T, H, W = D["logchl"].shape
    land = D["land"].astype(bool); ocean = ~land
    D["land"], D["ocean"] = land, ocean
    D["V"] = np.isfinite(D["logchl"]) & ocean[None]
    D["doy"], D["month"] = dates.dayofyear.values, dates.month.values
    sp = {k: np.where((dates >= a) & (dates <= b))[0] for k, (a, b) in cfg["SPLIT"].items()}
    D["sp"] = sp; tr = sp["train"]
    for k, v in sp.items():
        assert len(v) > 0, f"empty split {k}"
    assert sp["train"].max() < sp["val"].min() and sp["val"].max() < sp["test"].min(), "splits must be chronological"
    st = {}
    for v in ["logchl"] + ENV_VARS:
        x = D[v][tr][:, ocean]
        st[v] = (float(np.nanmean(x)), float(np.nanstd(x) + 1e-6))
    D["stats"] = st
    D["Z"] = ((D["logchl"] - st["logchl"][0]) / st["logchl"][1]).astype(np.float32)
    # climatology of normalised Chl (train only) - also a reconstruction baseline
    Zc = np.zeros((12, H, W), np.float32)
    for m in range(1, 13):
        idx = tr[D["month"][tr] == m]
        Zc[m - 1] = np.nan_to_num(np.nanmean(D["Z"][idx], 0)) if len(idx) else 0.0
    D["Zclim"] = Zc
    # ---- PFZ proxy: rolling window effort, threshold from TRAIN nonzero values only ----
    E = D["effort"].astype(np.float64); E[:, land] = 0
    a, b = cfg["PFZ_WINDOW"]
    cs = np.zeros((T + 1, H, W), np.float64); np.cumsum(E, 0, out=cs[1:])
    ti = np.arange(T)
    E8 = (cs[np.clip(ti + b, 0, T)] - cs[np.clip(ti + a, 0, T)]).astype(np.float32)
    del cs
    D["pfz_day_ok"] = (ti + a >= 0) & (ti + b <= T)
    fp = ((E[tr] > 0).sum(0) >= cfg["PFZ_FOOTPRINT_MIN_DAYS"]) & ocean
    vals = E8[tr][:, fp]; nz = vals[vals > 0]
    tau = float(np.quantile(nz, cfg["PFZ_QUANTILE"])) if len(nz) else 1.0
    D["Y"] = (E8 >= tau) & fp[None]
    D["fp"], D["pfz_tau"] = fp, tau
    prev = float(D["Y"][tr][:, fp].mean()) if fp.any() else float("nan")
    D["pfz_prev_train"] = prev
    pc = np.zeros((12, H, W), np.float32)
    for m in range(1, 13):
        idx = tr[(D["month"][tr] == m) & D["pfz_day_ok"][tr]]
        pc[m - 1] = D["Y"][idx].mean(0) if len(idx) else prev
    D["pclim"] = np.clip(pc, 0.005, 0.995) * fp
    # ---- static position features ----
    LAT, LON = np.meshgrid(D["lat"], D["lon"], indexing="ij")
    D["latn"] = ((LAT - LAT.mean()) / (LAT.std() + 1e-6)).astype(np.float32)
    D["lonn"] = ((LON - LON.mean()) / (LON.std() + 1e-6)).astype(np.float32)
    # ---- front threshold: 90th pct of |grad log10 Chl| on train days (fully valid 3x3) ----
    rng = np.random.default_rng(0); g = []
    for t in rng.choice(tr, min(200, len(tr)), replace=False):
        v = D["V"][t]
        if v.sum() < 50:
            continue
        gm = np.hypot(*sobel_np(np.where(v, D["logchl"][t], 0.0)))
        g.append(gm[valid3(v)])
    D["front_thr"] = float(np.quantile(np.concatenate(g), 0.90)) if g else 0.05
    # ---- data-driven 'monsoon' months = cloudiest calendar months in TRAIN ----
    miss = np.array([1 - D["V"][tr[D["month"][tr] == m]].sum() / max((D["month"][tr] == m).sum() * ocean.sum(), 1)
                     for m in range(1, 13)])
    D["monsoon_months"] = (np.argsort(-miss)[:cfg["N_MONSOON_MONTHS"]] + 1).tolist()
    D["train_days_ok"] = tr[D["V"][tr].reshape(len(tr), -1).sum(1) >= 0.05 * ocean.sum()]
    P = cfg["PATCH"]
    D["cands"] = np.array([(y, x) for y in tile_positions(H, P, 8) for x in tile_positions(W, P, 8)
                           if ocean[y:y + P, x:x + P].mean() >= cfg["MIN_PATCH_OCEAN"]])
    log(f"grid {H}x{W}, T={T}, ocean px={ocean.sum()}, split sizes " + str({k: len(v) for k, v in sp.items()}))
    log(f"Chl valid fraction (ocean): train {D['V'][tr][:, ocean].mean():.3f}")
    log(f"cloudiness by month (train): " + ", ".join(f"{m + 1}:{x:.2f}" for m, x in enumerate(miss)))
    log(f"'monsoon' (cloudiest) months: {D['monsoon_months']} | front thr {D['front_thr']:.4f} log10 units/cell")
    log(f"PFZ: tau={tau:.2f} h (8-day), footprint px={fp.sum()}, train prevalence in footprint={prev:.3f}")
    return D

In [ ]:
# ============================== MASKS ==============================
class MaskBank:
    """keep-masks (True = observation retained). 'cloud' transfers the real MODIS clear-sky pattern of another
    day (DINCAE/DINEOF-style); 'persistent' intersects clear-sky over PERSIST_DAYS consecutive days drawn from
    the cloudiest months and applies the SAME mask to target and context days; 'blob' is synthetic."""
    def __init__(self, D, cfg):
        self.V, self.cfg = D["V"], cfg; self.T = self.V.shape[0]
        self.offsets = [0] + list(cfg["CONTEXT_OFFSETS"])
        mon = D["month"]
        self.days = D["sp"]
        self.monsoon = {k: v[np.isin(mon[v], D["monsoon_months"])] for k, v in D["sp"].items()}

    def _w(self, a, win):
        if win is None:
            return a
        y0, x0, h, w = win
        return a[y0:y0 + h, x0:x0 + w]

    def sample(self, rng, kind, split, shape, win=None, drop=None, exclude=None):
        offs = self.offsets
        if kind == "random":
            p = drop if drop is not None else rng.uniform(*self.cfg["RANDOM_DROP"])
            return {o: rng.random(shape) >= p for o in offs}
        if kind == "blob":
            from scipy.ndimage import gaussian_filter
            sig, cov = rng.uniform(2, 8), rng.uniform(0.3, 0.9)
            base = gaussian_filter(rng.standard_normal(shape), sig)
            res = {}
            for o in offs:
                f = base if o == 0 else 0.8 * base + 0.2 * gaussian_filter(rng.standard_normal(shape), sig)
                res[o] = f > np.quantile(f, cov)
            return res
        P = self.cfg["PERSIST_DAYS"]
        if kind == "cloud":
            pool = self.days[split]
            pool = pool[(pool + min(offs) >= 0) & (pool + max(offs) < self.T)]
        else:
            pool = self.monsoon[split] if len(self.monsoon[split]) else self.days[split]
            pool = pool[pool + P - 1 < self.T]
        if exclude is not None:
            pool = pool[np.abs(pool - exclude) > 1]
        d = int(rng.choice(pool))
        if kind == "cloud":
            return {o: self._w(self.V[d + o], win).copy() for o in offs}
        keep = np.ones(shape, bool)
        for j in range(P):
            keep &= self._w(self.V[d + j], win)
        return {o: keep for o in offs}

def build_eval_set(D, bank, cfg, split, n_days, kinds, seed):
    V, ocean = D["V"], D["ocean"]; T, H, W = V.shape; offs = bank.offsets
    ok = np.array([t for t in D["sp"][split]
                   if V[t].sum() >= cfg["EVAL_MIN_VALID_FRAC"] * ocean.sum() and all(0 <= t + o < T for o in offs)])
    assert len(ok) > 0, f"no eligible {split} days - lower EVAL_MIN_VALID_FRAC"
    sel = np.unique(ok[np.linspace(0, len(ok) - 1, min(n_days, len(ok))).round().astype(int)])
    masks = {}
    for kind in kinds:
        arr = np.zeros((len(sel), len(offs), H, W), bool)
        for i, t in enumerate(sel):
            rng = np.random.default_rng([seed, int(t), KIND_ID[kind]])
            for _ in range(20):
                kd = bank.sample(rng, kind, split, (H, W), None, drop=cfg["EVAL_RANDOM_DROP"], exclude=int(t))
                if (V[t] & ~kd[0]).sum() >= 0.1 * V[t].sum():
                    break
            arr[i] = np.stack([kd[o] for o in offs])
        masks[kind] = arr
        hf = np.mean([(V[t] & ~arr[i, 0]).sum() / V[t].sum() for i, t in enumerate(sel)])
        log(f"eval[{split}] {kind:10s}: {len(sel)} days, mean hidden fraction of valid px = {hf:.2f}")
    return dict(days=sel, masks=masks)

def save_eval_sets(path, evs):
    flat = {}
    for name, ev in evs.items():
        flat[f"{name}__days"] = ev["days"]
        for k, m in ev["masks"].items():
            flat[f"{name}__{k}"] = np.packbits(m, axis=-1); flat[f"{name}__{k}__shape"] = np.array(m.shape)
    np.savez_compressed(path, **flat)

def load_eval_sets(path):
    z = np.load(path); evs = {}
    for key in z.files:
        if key.endswith("__days"):
            name = key[:-6]; masks = {}
            for k2 in z.files:
                if k2.startswith(name + "__") and not k2.endswith("__days") and not k2.endswith("__shape"):
                    kind = k2[len(name) + 2:]; shp = tuple(z[k2 + "__shape"])
                    masks[kind] = np.unpackbits(z[k2], axis=-1, count=shp[-1]).astype(bool).reshape(shp)
            evs[name] = dict(days=z[key], masks=masks)
    return evs

In [ ]:
# ============================== MODEL INPUTS (shared by training, inference, baselines) ==============================
def n_cond_channels(cfg):
    return 2 + 2 * len(cfg["CONTEXT_OFFSETS"]) + len(ENV_VARS) + 3

def build_inputs(D, t, keeps, win, cfg):
    """cond = [obs Chl_t, mask_t, (obs Chl_{t+o}, mask_{t+o})..., SST, SLA, u, v, land, lat, lon].
    Hidden truth never enters cond. x0 is returned only for losses/metrics/compositing observed pixels."""
    Z, V = D["Z"], D["V"]; T = Z.shape[0]
    if win is None:
        ys = xs = slice(None)
    else:
        y0, x0, h, w = win; ys, xs = slice(y0, y0 + h), slice(x0, x0 + w)
    obs0 = V[t, ys, xs] & keeps[0]
    z0 = np.nan_to_num(Z[t, ys, xs])
    ch = [np.where(obs0, z0, 0.0), obs0]
    for j, o in enumerate(cfg["CONTEXT_OFFSETS"], start=1):
        tt = t + o
        if 0 <= tt < T:
            m = V[tt, ys, xs] & keeps[j]
            ch += [np.where(m, np.nan_to_num(Z[tt, ys, xs]), 0.0), m]
        else:
            ch += [np.zeros_like(z0), np.zeros_like(obs0)]
    for v in ENV_VARS:
        mu, sd = D["stats"][v]
        ch.append(np.nan_to_num((D[v][t, ys, xs] - mu) / sd))
    ch += [D["land"][ys, xs], D["latn"][ys, xs], D["lonn"][ys, xs]]
    cond = np.stack([np.asarray(c, np.float32) for c in ch])
    ang = 2 * np.pi * D["doy"][t] / 365.25
    return dict(cond=cond, extra=np.array([np.sin(ang), np.cos(ang)], np.float32), obs=obs0,
                x0=z0.astype(np.float32), valid=V[t, ys, xs],
                pstat=D["pclim"][D["month"][t] - 1][ys, xs][None].astype(np.float32))

def make_train_sample(D, bank, cfg, rng):
    P = cfg["PATCH"]; offs = bank.offsets
    kinds = list(cfg["TRAIN_MASK_PROBS"]); pr = np.array([cfg["TRAIN_MASK_PROBS"][k] for k in kinds], float); pr /= pr.sum()
    found = False
    for _ in range(50):
        t = int(rng.choice(D["train_days_ok"])); y0, x0 = D["cands"][rng.integers(len(D["cands"]))]
        win = (int(y0), int(x0), P, P)
        Vp = D["V"][t, y0:y0 + P, x0:x0 + P]
        if Vp.sum() < cfg["MIN_PATCH_VALID"] * D["ocean"][y0:y0 + P, x0:x0 + P].sum():
            continue
        kind = kinds[rng.choice(len(kinds), p=pr)]
        kd = bank.sample(rng, kind, "train", (P, P), win, exclude=t)
        keeps = [kd[o] for o in offs]
        if (Vp & ~keeps[0]).sum() >= 16:
            found = True; break
    if not found:
        keeps = [rng.random((P, P)) >= 0.5 for _ in offs]
    inp = build_inputs(D, t, keeps, win, cfg)
    sl = (slice(y0, y0 + P), slice(x0, x0 + P))
    valid, obs = inp["valid"], inp["obs"]
    if t - 1 >= 0:
        prev, vprev = np.nan_to_num(D["Z"][t - 1][sl]), D["V"][t - 1][sl]
        coslat = np.cos(np.deg2rad(D["lat"][y0:y0 + P]))[:, None]
        dxp = np.nan_to_num(D["ugos"][t][sl]) * 86400 / (cfg["GRID_RES"] * 111320 * coslat)
        dyp = np.nan_to_num(D["vgos"][t][sl]) * 86400 / (cfg["GRID_RES"] * 110540)
    else:
        prev, vprev = np.zeros((P, P)), np.zeros((P, P), bool); dxp = dyp = np.zeros((P, P))
    f32 = lambda a: np.asarray(a, np.float32)[None]
    return dict(x0=f32(inp["x0"]), cond=inp["cond"], extra=inp["extra"], pstat=inp["pstat"],
                valid=f32(valid), obs=f32(obs), hidden=f32(valid & ~obs), ocean=f32(D["ocean"][sl]),
                y=f32(D["Y"][t][sl]), pw=f32(D["fp"][sl] & D["pfz_day_ok"][t]),
                prev=f32(prev), vprev=f32(vprev), disp=np.stack([dxp, dyp]).astype(np.float32))

In [ ]:
# ============================== METRICS ==============================
def crps_ensemble(X, y):
    """Fair ensemble CRPS. X: (N, M) members, y: (M,)"""
    N = X.shape[0]; t1 = np.abs(X - y[None]).mean(0)
    if N == 1:
        return t1
    Xs = np.sort(X, 0); i = np.arange(1, N + 1)[:, None]
    return t1 - 0.5 * (2 * ((2 * i - N - 1) * Xs).sum(0) / (N * (N - 1)))

def ssim_map(a, b, L=3.0, sigma=1.5):
    from scipy.ndimage import gaussian_filter as gf
    C1, C2 = (0.01 * L) ** 2, (0.03 * L) ** 2
    ma, mb = gf(a, sigma), gf(b, sigma)
    va, vb, cab = gf(a * a, sigma) - ma ** 2, gf(b * b, sigma) - mb ** 2, gf(a * b, sigma) - ma * mb
    return ((2 * ma * mb + C1) * (2 * cab + C2)) / ((ma ** 2 + mb ** 2 + C1) * (va + vb + C2))

def radial_psd(f):
    n = f.shape[0]; f = f - f.mean(); w = np.hanning(n); f = f * np.outer(w, w)
    Pw = np.abs(np.fft.fftshift(np.fft.fft2(f))) ** 2
    ky, kx = np.indices(Pw.shape) - n // 2; k = np.hypot(kx, ky).round().astype(int)
    prof = np.bincount(k.ravel(), Pw.ravel()) / np.maximum(np.bincount(k.ravel()), 1)
    return prof[1:n // 2]

def ece_score(p, y, bins=15):
    idx = np.clip((p * bins).astype(int), 0, bins - 1); e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def reliability(p, y, bins=10):
    idx = np.clip((p * bins).astype(int), 0, bins - 1)
    return np.array([[p[idx == b].mean(), y[idx == b].mean(), (idx == b).sum()] if (idx == b).any()
                     else [np.nan, np.nan, 0] for b in range(bins)])

def choose_theta(p, y):
    y = y.astype(bool); best = (0.5, -1)
    for th in np.arange(0.02, 0.99, 0.02):
        a = p >= th; iou = (a & y).sum() / max((a | y).sum(), 1)
        if iou > best[1]:
            best = (float(th), float(iou))
    return best[0]

def pfz_metrics(p, y, pclim, theta=None):
    from sklearn.metrics import roc_auc_score, average_precision_score
    y = y.astype(bool); p = np.clip(p.astype(np.float64), 1e-6, 1 - 1e-6)
    out = dict(n=int(len(y)), prevalence=float(y.mean()))
    if 0 < y.sum() < len(y):
        out["auroc"] = float(roc_auc_score(y, p)); out["auprc"] = float(average_precision_score(y, p))
    out["brier"] = float(np.mean((p - y) ** 2)); bc = float(np.mean((pclim - y) ** 2))
    out["bss_vs_clim"] = float(1 - out["brier"] / bc) if bc > 0 else float("nan")
    out["logloss"] = float(-np.mean(y * np.log(p) + (~y) * np.log(1 - p)))
    out["ece"] = ece_score(p, y.astype(float))
    if theta is not None:
        a = p >= theta; out["iou"] = float((a & y).sum() / max((a | y).sum(), 1)); out["theta"] = float(theta)
    return out

In [ ]:
# ============================== EVALUATION (model-agnostic) ==============================
def evaluate(name, predict_fn, D, ev, cfg, n_main, n_uq=0, uq_days=0, theta=None, kinds=None):
    """predict_fn(inp, n, kind, t) -> (ens (N,H,W) normalised log-Chl or None, pfz probs (N,H,W) or None)"""
    mu, sd = D["stats"]["logchl"]; thr = D["front_thr"]; ocean = D["ocean"]; land = ~ocean
    H, W = ocean.shape; rows, arrays, pfz_sum = [], {}, {}
    for kind in (kinds or list(ev["masks"])):
        M = ev["masks"][kind]; Pl, Yl, Sl, Cl, ue, us, pt, pm, ps = [], [], [], [], [], [], [], [], []
        arrays[f"{kind}__map_days"] = np.array(ev["days"][:cfg["KEEP_MAPS"]])
        for i, t in enumerate(ev["days"]):
            t = int(t); inp = build_inputs(D, t, list(M[i]), None, cfg)
            n = n_uq if (i < uq_days and n_uq > n_main) else n_main
            tic = time.time(); ens, pz = predict_fn(inp, n, kind, t); dt = time.time() - tic
            row = dict(model=name, kind=kind, day=t, date=str(D["dates"][t]), sec=round(dt, 2))
            valid, obs = D["V"][t], inp["obs"]; hidden = valid & ~obs
            if ens is not None:
                ens = np.where(obs[None], inp["x0"][None], ens)
                E = ens * sd + mu; Em = E[:n_main]; mean = Em.mean(0); truth = D["logchl"][t]
                y = truth[hidden]; err = mean[hidden] - y
                row.update(n_hidden=int(hidden.sum()), rmse=float(np.sqrt(np.mean(err ** 2))),
                           mae=float(np.mean(np.abs(err))), bias=float(np.mean(err)),
                           r2=float(1 - np.sum(err ** 2) / max(np.sum((y - y.mean()) ** 2), 1e-12)),
                           crps=float(crps_ensemble(Em[:, hidden], y).mean()))
                if Em.shape[0] > 1:
                    Nn = Em.shape[0]; spr = float(np.sqrt(np.mean(Em.var(0, ddof=1)[hidden])))
                    row["spread"] = spr; row["ssr"] = spr * np.sqrt((Nn + 1) / Nn) / max(row["rmse"], 1e-9)
                    if Nn >= 10:
                        q = np.quantile(Em[:, hidden], [0.05, 0.95], axis=0)
                        row["cov90"] = float(np.mean((y >= q[0]) & (y <= q[1])))
                gt = np.hypot(*sobel_np(np.where(valid, truth, 0.0))); elig = hidden & valid3(valid)
                for tag, fld in (("mean", mean), ("s0", Em[0])):
                    if elig.sum() > 0:
                        gp = np.hypot(*sobel_np(np.where(ocean, fld, 0.0)))
                        row[f"grad_mae_{tag}"] = float(np.abs(gp - gt)[elig].mean())
                        a_, b_ = gt[elig] > thr, gp[elig] > thr
                        row[f"front_f1_{tag}"] = float(2 * (a_ & b_).sum() / max(a_.sum() + b_.sum(), 1)) if a_.any() else np.nan
                A_ = np.where(land, 0.0, mean); B_ = np.where(land, 0.0, np.where(valid, truth, mean))
                row["ssim"] = float(ssim_map(A_, B_)[hidden].mean())
                w_ = cfg["PSD_WIN"]
                for y0 in range(0, H - w_ + 1, w_ // 2):
                    for x0 in range(0, W - w_ + 1, w_ // 2):
                        s_ = (slice(y0, y0 + w_), slice(x0, x0 + w_))
                        if valid[s_].mean() >= 0.95 and hidden[s_].mean() >= 0.25:
                            pt.append(radial_psd(np.where(valid[s_], truth[s_], mean[s_])))
                            pm.append(radial_psd(mean[s_])); ps.append(radial_psd(Em[0][s_]))
                if i < uq_days and E.shape[0] > 1:
                    ue.append(np.abs(E.mean(0)[hidden] - y)); us.append(E.std(0, ddof=1)[hidden])
                if i < cfg["KEEP_MAPS"]:
                    arrays[f"{kind}__map{i}__mean"] = mean.astype(np.float16)
                    arrays[f"{kind}__map{i}__s0"] = Em[0].astype(np.float16)
                    arrays[f"{kind}__map{i}__std"] = (Em.std(0) if Em.shape[0] > 1 else np.zeros_like(mean)).astype(np.float16)
            if pz is not None and D["pfz_day_ok"][t]:
                fp = D["fp"]; Pm = pz.mean(0); yb = D["Y"][t][fp]
                Pl.append(Pm[fp]); Yl.append(yb); Sl.append((pz.std(0) if pz.shape[0] > 1 else np.zeros_like(Pm))[fp])
                Cl.append(D["pclim"][D["month"][t] - 1][fp])
                row["pfz_brier"] = float(np.mean((Pm[fp] - yb) ** 2))
                if theta is not None:
                    a_ = Pm[fp] >= theta; row["pfz_iou"] = float((a_ & yb).sum() / max((a_ | yb).sum(), 1))
                if i < cfg["KEEP_MAPS"]:
                    arrays[f"{kind}__map{i}__pfz"] = Pm.astype(np.float16)
                    arrays[f"{kind}__map{i}__pfzstd"] = (pz.std(0) if pz.shape[0] > 1 else np.zeros_like(Pm)).astype(np.float16)
            rows.append(row)
            if i == 0:
                log(f"  [{name}|{kind}] first case took {dt:.1f}s -> ~{dt * len(ev['days']) / 60:.1f} min for this mask type")
        if Pl:
            p, yb, s, c = map(np.concatenate, (Pl, Yl, Sl, Cl))
            pfz_sum[kind] = pfz_metrics(p, yb, c, theta)
            arrays[f"{kind}__pfz_p"] = p.astype(np.float16); arrays[f"{kind}__pfz_y"] = yb
            arrays[f"{kind}__pfz_s"] = s.astype(np.float16); arrays[f"{kind}__pfz_c"] = c.astype(np.float16)
        if ue:
            arrays[f"{kind}__uq_err"] = np.concatenate(ue).astype(np.float32)
            arrays[f"{kind}__uq_std"] = np.concatenate(us).astype(np.float32)
        if pt:
            for nm, L_ in (("true", pt), ("mean", pm), ("s0", ps)):
                arrays[f"{kind}__psd_{nm}"] = np.log10(np.mean(L_, 0) + 1e-12); arrays[f"{kind}__psd_n"] = np.array(len(pt))
    return dict(rows=pd.DataFrame(rows), pfz=pfz_sum, arrays=arrays)

def save_eval(run_dir, res, meta=None):
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    res["rows"].to_csv(run_dir / "metrics_rows.csv", index=False)
    (run_dir / "pfz_summary.json").write_text(json.dumps(res["pfz"], indent=1))
    np.savez_compressed(run_dir / "eval_arrays.npz", **res["arrays"])
    if meta:
        (run_dir / "meta.json").write_text(json.dumps(meta, indent=1, default=str))

def theta_from_val(name, predict_fn, D, ev_val, cfg, n):
    r = evaluate(name + "[val]", predict_fn, D, ev_val, cfg, n_main=n)
    k = [k for k in r["arrays"] if k.endswith("__pfz_p")]
    if not k:
        return None
    p = r["arrays"][k[0]].astype(np.float32); y = r["arrays"][k[0][:-2] + "_y"]
    th = choose_theta(p, y); log(f"  {name}: theta (max IoU on val) = {th:.2f}")
    return th

In [ ]:
# ============================== DINEOF (numpy re-implementation, Beckers & Rixen-style) ==============================
def _svd_recon(A, k, seed):
    from sklearn.utils.extmath import randomized_svd
    U, s, Vt = randomized_svd(A, n_components=k, n_iter=4, random_state=seed)
    return (U * s) @ Vt

def dineof(X, kmax=30, max_iter=40, tol=1e-3, cv_frac=0.02, seed=0):
    """X (n_time, n_space) with NaN gaps. Mode count chosen by cross-validation on held-out observed points."""
    rng = np.random.default_rng(seed); X = X.astype(np.float32); miss = ~np.isfinite(X)
    colmean = np.nan_to_num(np.nanmean(X, 0)).astype(np.float32)
    A = np.where(miss, 0.0, X - colmean).astype(np.float32)
    obs = np.flatnonzero(~miss); cv = rng.choice(obs, max(100, int(cv_frac * len(obs))), replace=False)
    cv_true = A.flat[cv].copy(); A.flat[cv] = 0.0; fill = miss.copy(); fill.flat[cv] = True
    best_err, best_k, best_A, errs = np.inf, 1, None, []
    for k in range(1, kmax + 1):
        for _ in range(max_iter):
            R = _svd_recon(A, k, seed); d = np.sqrt(np.mean((R[fill] - A[fill]) ** 2)); A[fill] = R[fill]
            if d < tol:
                break
        err = float(np.sqrt(np.mean((A.flat[cv] - cv_true) ** 2))); errs.append(err)
        if err < best_err:
            best_err, best_k, best_A = err, k, A.copy()
        elif k - best_k >= 3:
            break
    A = best_A; A.flat[cv] = cv_true
    for _ in range(max_iter):
        R = _svd_recon(A, best_k, seed); d = np.sqrt(np.mean((R[miss] - A[miss]) ** 2)); A[miss] = R[miss]
        if d < tol:
            break
    return A + colmean, best_k, errs

def run_dineof(D, ev, cfg, split="test"):
    """Transductive, as DINEOF is normally used: sees the whole split's time series (with eval masks applied to
    eval days and their context days). Document this asymmetry in the paper."""
    days = D["sp"][split]; pos = {int(t): j for j, t in enumerate(days)}; ocean = D["ocean"]
    offs = [0] + list(cfg["CONTEXT_OFFSETS"]); out = {}
    for kind, M in ev["masks"].items():
        X = D["Z"][days][:, ocean].copy()
        for i, t in enumerate(ev["days"]):
            for j, o in enumerate(offs):
                if int(t) + o in pos:
                    X[pos[int(t) + o], ~M[i, j][ocean]] = np.nan
        tic = time.time(); R, k, errs = dineof(X, cfg["DINEOF_KMAX"], cfg["DINEOF_MAXITER"])
        log(f"DINEOF[{kind}] k*={k}, CV-RMSE(norm)={min(errs):.3f}, {time.time() - tic:.0f}s")
        out[kind] = {int(t): R[pos[int(t)]] for t in ev["days"]}
    return out

In [ ]:
# [TORCH] ============================== MODELS ==============================
import torch
import torch.nn as nn
import torch.nn.functional as F
HAS_TORCH = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = bool(CFG["AMP"] and DEVICE.type == "cuda")
torch.backends.cudnn.benchmark = True
log("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "", "| AMP:", AMP, "| torch", torch.__version__)

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP) if DEVICE.type == "cuda" else contextlib.nullcontext()

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=AMP)

def sinusoidal(t, dim):
    half = dim // 2
    f = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    a = t.float()[:, None] * f[None]
    return torch.cat([torch.sin(a), torch.cos(a)], 1)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, ed, dropout, g=8):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(g, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.emb = nn.Linear(ed, 2 * cout)
        self.n2, self.drop, self.c2 = nn.GroupNorm(g, cout), nn.Dropout(dropout), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, emb):
        h = self.c1(F.silu(self.n1(x)))
        s, b = self.emb(F.silu(emb))[:, :, None, None].chunk(2, 1)
        h = self.n2(h) * (1 + s) + b
        return self.c2(self.drop(F.silu(h))) + self.skip(x)

class Attn(nn.Module):
    def __init__(self, ch, g=8, heads=4):
        super().__init__()
        self.norm, self.qkv, self.proj, self.h = nn.GroupNorm(g, ch), nn.Conv2d(ch, 3 * ch, 1), nn.Conv2d(ch, ch, 1), heads
    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).reshape(B, 3, self.h, C // self.h, H * W).unbind(1)
        o = F.scaled_dot_product_attention(q.transpose(-1, -2), k.transpose(-1, -2), v.transpose(-1, -2))
        return x + self.proj(o.transpose(-1, -2).reshape(B, C, H, W))

class Down(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.c = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.c(x)

class Up(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.c = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.c(F.interpolate(x, scale_factor=2, mode="nearest"))

class UNet(nn.Module):
    """FiLM-conditioned U-Net. use_t=True -> diffusion denoiser; False -> deterministic net (baseline / PFZ head)."""
    def __init__(self, in_ch, out_ch, base=64, mult=(1, 2, 2, 2), n_res=2, attn_res=(8,), dropout=0.1,
                 use_t=True, n_extra=2, patch=64, g=8):
        super().__init__()
        ed = base * 4; self.base, self.use_t = base, use_t
        self.t_mlp = nn.Sequential(nn.Linear(base, ed), nn.SiLU(), nn.Linear(ed, ed)) if use_t else None
        self.x_mlp = nn.Sequential(nn.Linear(n_extra, ed), nn.SiLU(), nn.Linear(ed, ed))
        self.inc = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down, chans, ch, res = nn.ModuleList(), [base], base, patch
        for i, m in enumerate(mult):
            for _ in range(n_res):
                blk = nn.ModuleList([ResBlock(ch, base * m, ed, dropout, g)]); ch = base * m
                if res in attn_res:
                    blk.append(Attn(ch, g))
                self.down.append(blk); chans.append(ch)
            if i < len(mult) - 1:
                self.down.append(nn.ModuleList([Down(ch)])); chans.append(ch); res //= 2
        self.mid = nn.ModuleList([ResBlock(ch, ch, ed, dropout, g), Attn(ch, g), ResBlock(ch, ch, ed, dropout, g)])
        self.up = nn.ModuleList()
        for i, m in reversed(list(enumerate(mult))):
            for _ in range(n_res + 1):
                blk = nn.ModuleList([ResBlock(ch + chans.pop(), base * m, ed, dropout, g)]); ch = base * m
                if res in attn_res:
                    blk.append(Attn(ch, g))
                self.up.append(blk)
            if i > 0:
                self.up.append(nn.ModuleList([Up(ch)])); res *= 2
        self.out = nn.Sequential(nn.GroupNorm(g, ch), nn.SiLU(), nn.Conv2d(ch, out_ch, 3, padding=1))

    def forward(self, x, t=None, extra=None):
        emb = self.x_mlp(extra)
        if self.use_t:
            emb = emb + self.t_mlp(sinusoidal(t, self.base))
        h = self.inc(x); hs = [h]
        for blk in self.down:
            if isinstance(blk[0], Down):
                h = blk[0](h)
            else:
                h = blk[0](h, emb)
                if len(blk) > 1:
                    h = blk[1](h)
            hs.append(h)
        h = self.mid[2](self.mid[1](self.mid[0](h, emb)), emb)
        for blk in self.up:
            if isinstance(blk[0], Up):
                h = blk[0](h)
            else:
                h = blk[0](torch.cat([h, hs.pop()], 1), emb)
                if len(blk) > 1:
                    h = blk[1](h)
        return self.out(h)

def sobel_xy(x):
    kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=x.device, dtype=x.dtype).view(1, 1, 3, 3) / 8
    xp = F.pad(x, (1, 1, 1, 1), mode="replicate")
    return F.conv2d(xp, kx), F.conv2d(xp, kx.transpose(2, 3))

PFZ_IN_CH = 12
def pfz_features(chl, obs, cond, pstat, nctx):
    """[Chl composite, obs mask, |grad Chl|, SST, |grad SST|, SLA, u, v, land, lat, lon, monthly PFZ climatology]"""
    b = 2 + 2 * nctx; env = cond[:, b:b + 4]
    gx, gy = sobel_xy(chl); gc = torch.sqrt(gx ** 2 + gy ** 2 + 1e-6)
    sx, sy = sobel_xy(env[:, :1]); gs = torch.sqrt(sx ** 2 + sy ** 2 + 1e-6)
    return torch.cat([chl, obs, 10 * gc, env[:, :1], 10 * gs, env[:, 1:4], cond[:, b + 4:b + 7], pstat], 1)

class CODA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        P = cfg["PATCH"]; self.nctx = len(cfg["CONTEXT_OFFSETS"])
        self.den = UNet(1 + n_cond_channels(cfg), 1, cfg["BASE_CH"], cfg["CH_MULT"], cfg["N_RES"], cfg["ATTN_RES"],
                        cfg["DROPOUT"], use_t=True, patch=P)
        self.pfz = UNet(PFZ_IN_CH, 1, cfg["PFZ_BASE_CH"], (1, 2, 2), 1, (), 0.1, use_t=False, patch=P)

class GaussUNet(nn.Module):     # DINCAE-style deterministic baseline: same backbone, outputs mean + log-variance
    def __init__(self, cfg):
        super().__init__()
        self.net = UNet(n_cond_channels(cfg), 2, cfg["BASE_CH"], cfg["CH_MULT"], cfg["N_RES"], cfg["ATTN_RES"],
                        cfg["DROPOUT"], use_t=False, patch=cfg["PATCH"])

class EnvPFZ(nn.Module):        # PFZ from SST/SLA/currents/position/climatology only (no Chl)
    def __init__(self, cfg):
        super().__init__(); self.nctx = len(cfg["CONTEXT_OFFSETS"])
        self.net = UNet(PFZ_IN_CH, 1, cfg["PFZ_BASE_CH"], (1, 2, 2), 1, (), 0.1, use_t=False, patch=cfg["PATCH"])
    def feats(self, cond, pstat):
        z = torch.zeros_like(cond[:, :1]); return pfz_features(z, z, cond, pstat, self.nctx)

def alpha_bar(T, s=0.008):
    x = torch.arange(T + 1, dtype=torch.float64) / T
    f = torch.cos((x + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.99999).float()

AB = alpha_bar(CFG["T_DIFF"]).to(DEVICE)
_m = CODA(CFG); log(f"CODA params: denoiser {sum(p.numel() for p in _m.den.parameters()) / 1e6:.2f}M, "
                    f"PFZ head {sum(p.numel() for p in _m.pfz.parameters()) / 1e6:.2f}M"); del _m

In [ ]:
# [TORCH] ============================== LOSSES ==============================
def disp_grid(disp):
    B, _, H, W = disp.shape
    yy, xx = torch.meshgrid(torch.arange(H, device=disp.device, dtype=disp.dtype),
                            torch.arange(W, device=disp.device, dtype=disp.dtype), indexing="ij")
    sx, sy = xx[None] - disp[:, 0], yy[None] - disp[:, 1]
    return torch.stack([2 * sx / (W - 1) - 1, 2 * sy / (H - 1) - 1], -1)

def coda_losses(model, b, cfg, mode, pos_weight):
    x0, cond, extra = b["x0"], b["cond"], b["extra"]
    valid, obs, hidden, ocean = b["valid"], b["obs"], b["hidden"], b["ocean"]
    B, T = x0.shape[0], AB.shape[0]; tmax = max(1, int(cfg["PFZ_TMAX_FRAC"] * T))
    t = torch.randint(0, T, (B,), device=x0.device)
    nlow = int(cfg["LOWT_FRAC"] * B)
    if nlow > 0:          # over-sample low-noise steps so the recon / PFZ terms get signal every batch
        t[:nlow] = torch.randint(0, tmax, (nlow,), device=x0.device)
    a, s = AB[t].sqrt().view(B, 1, 1, 1), (1 - AB[t]).sqrt().view(B, 1, 1, 1)
    eps = torch.randn_like(x0); xt = a * x0 + s * eps
    with autocast_ctx():
        v_hat = model.den(torch.cat([xt, cond], 1), t, extra)
    v_hat = v_hat.float(); L = {}
    w = valid * (hidden + cfg["W_OBSERVED"] * obs)            # no loss where truth is missing (cloud in original data)
    L["diff"] = (w * (v_hat - (a * eps - s * x0)) ** 2).sum() / (w.sum() + 1)
    x0_hat = (a * xt - s * v_hat).clamp(-8, 8)                  # one-step clean estimate -> differentiable
    comp = obs * x0 + (1 - obs) * x0_hat
    low = (t < tmax).float().view(B, 1, 1, 1); m = low * hidden
    L["recon"] = (m * (x0_hat - x0).abs()).sum() / (m.sum() + 1)
    gm = m * (F.avg_pool2d(valid, 3, 1, 1) > 0.999).float()
    gxh, gyh = sobel_xy(comp); gxt, gyt = sobel_xy(x0 * valid)
    L["grad"] = (gm * ((gxh - gxt).abs() + (gyh - gyt).abs())).sum() / (gm.sum() + 1)
    if cfg["LAMBDA"].get("adv", 0) > 0:      # weak supervision where there is NO truth at t: advected truth of t-1
        grid = disp_grid(b["disp"])
        pa = F.grid_sample(b["prev"], grid, mode="bilinear", padding_mode="zeros", align_corners=True)
        va = F.grid_sample(b["vprev"], grid, mode="bilinear", padding_mode="zeros", align_corners=True)
        am = low * ocean * (1 - valid) * (va > 0.99).float()
        L["adv"] = (am * (x0_hat - pa).abs()).sum() / (am.sum() + 1)
    chl_in = comp if mode == "coupled" else comp.detach()     # <-- the ONLY difference between coupled / uncoupled
    feats = pfz_features(chl_in, obs, cond, b["pstat"], model.nctx)
    with autocast_ctx():
        logits = model.pfz(feats, None, extra)
    bce = F.binary_cross_entropy_with_logits(logits.float(), b["y"], reduction="none", pos_weight=pos_weight)
    pm = low * b["pw"]
    L["pfz"] = (pm * bce).sum() / (pm.sum() + 1)
    return L

def coda_loss_fn(mode):
    pw = torch.tensor([CFG["PFZ_POS_WEIGHT"]], device=DEVICE)
    return lambda model, b: {k: CFG["LAMBDA"][k] * v for k, v in coda_losses(model, b, CFG, mode, pw).items()}

def gauss_loss_fn(model, b):
    with autocast_ctx():
        out = model.net(b["cond"], None, b["extra"])
    out = out.float(); mu, lv = out[:, :1], out[:, 1:2].clamp(-8, 4)
    w = b["valid"] * (b["hidden"] + CFG["W_OBSERVED"] * b["obs"])
    return {"nll": (w * 0.5 * (lv + (b["x0"] - mu) ** 2 * torch.exp(-lv))).sum() / (w.sum() + 1)}

def envpfz_loss_fn(model, b):
    f = model.feats(b["cond"], b["pstat"])
    with autocast_ctx():
        lg = model.net(f, None, b["extra"])
    bce = F.binary_cross_entropy_with_logits(lg.float(), b["y"], reduction="none")
    return {"pfz": (b["pw"] * bce).sum() / (b["pw"].sum() + 1)}

In [ ]:
# [TORCH] ============================== TRAINER (EMA, AMP, time budget, resume) ==============================
class TrainPatches(torch.utils.data.Dataset):
    def __init__(self, D, bank, cfg, n, seed, offset=0):
        self.D, self.bank, self.cfg, self.n, self.seed, self.offset = D, bank, cfg, n, seed, offset
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        s = make_train_sample(self.D, self.bank, self.cfg, np.random.default_rng([self.seed, self.offset + i]))
        return {k: torch.from_numpy(np.ascontiguousarray(v)) for k, v in s.items()}

class EMA:
    def __init__(self, model, decay):
        self.decay, self.shadow = decay, copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))
        for ps, p in zip(self.shadow.parameters(), model.parameters()):
            ps.mul_(d).add_(p.detach(), alpha=1 - d)
        for bs, bb in zip(self.shadow.buffers(), model.buffers()):
            bs.copy_(bb)

def find_resume(run_name):
    hits = sorted(glob.glob(f"/kaggle/input/**/runs/{run_name}/ckpt.pt", recursive=True))
    return Path(hits[-1]) if hits else None

def fit(model, loss_fn, steps, run_dir, tag, seed):
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True); set_seed(seed)
    model = model.to(DEVICE); ema = EMA(model, CFG["EMA"])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=1e-4); scaler = make_scaler()
    step, hist = 0, []
    src = run_dir / "ckpt.pt" if (run_dir / "ckpt.pt").exists() else find_resume(run_dir.name)
    if src is not None:
        st = torch.load(src, map_location=DEVICE, weights_only=False)
        model.load_state_dict(st["model"]); ema.shadow.load_state_dict(st["ema"]); opt.load_state_dict(st["opt"])
        scaler.load_state_dict(st["scaler"]); step, hist = st["step"], st["hist"]; log(f"[{tag}] resumed from {src} @ step {step}")

    def save():
        torch.save(dict(model=model.state_dict(), ema=ema.shadow.state_dict(), opt=opt.state_dict(),
                        scaler=scaler.state_dict(), step=step, hist=hist), run_dir / "ckpt.pt")
        pd.DataFrame(hist).to_csv(run_dir / "train_hist.csv", index=False)

    if step >= steps:
        return ema.shadow.eval(), hist, True
    ds = TrainPatches(D, BANK, CFG, n=(steps - step) * CFG["BATCH"], seed=seed, offset=step * CFG["BATCH"])
    nw = min(CFG["WORKERS"], os.cpu_count() or 1)
    dl = torch.utils.data.DataLoader(ds, batch_size=CFG["BATCH"], shuffle=False, drop_last=True, num_workers=nw,
                                     pin_memory=DEVICE.type == "cuda", persistent_workers=nw > 0)
    t_end, t0, acc, n_acc = time.time() + CFG["MAX_TRAIN_MINUTES"] * 60, time.time(), {}, 0
    model.train()
    for batch in dl:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        lr = CFG["LR"] * min(1.0, (step + 1) / CFG["WARMUP"]) * max(0.05, 0.5 * (1 + math.cos(math.pi * step / steps)))
        for gp in opt.param_groups:
            gp["lr"] = lr
        parts = loss_fn(model, batch); loss = sum(parts.values())
        opt.zero_grad(set_to_none=True)
        if torch.isfinite(loss):
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            ema.update(model, step)
            for k, v in parts.items():
                acc[k] = acc.get(k, 0.0) + v.detach()
            n_acc += 1
        else:
            log(f"[{tag}] non-finite loss at step {step}, batch skipped")
        step += 1
        if step % CFG["LOG_EVERY"] == 0 and n_acc:
            rec = {"step": step, "lr": lr, "min": round((time.time() - t0) / 60, 2),
                   **{k: float(v) / n_acc for k, v in acc.items()}}
            hist.append(rec); acc, n_acc = {}, 0; log(f"[{tag}]", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rec.items()})
        stop = step >= steps or time.time() > t_end
        if step % CFG["CKPT_EVERY"] == 0 or stop:
            save()
        if stop:
            break
    save(); done = step >= steps
    if not done:
        log(f"[{tag}] TIME BUDGET HIT at step {step}/{steps}. Save this version, attach its output as input next session to resume.")
    return ema.shadow.eval(), hist, done

In [ ]:
# [TORCH] ============================== INFERENCE: tiled DDIM sampling + predictors ==============================
def _blend(tile):
    w = torch.sin(math.pi * (torch.arange(tile, device=DEVICE, dtype=torch.float32) + 0.5) / tile).clamp_min(0.05)
    return (w[:, None] * w[None, :])[None, None]

@torch.no_grad()
def tiled(fn, x, tile, stride, max_batch=256):
    """Run a patch-trained net over the full domain with overlapping, smoothly blended tiles (applied at EVERY
    DDIM step, so each sample stays globally coherent rather than being stitched afterwards)."""
    B, C, H0, W0 = x.shape
    ph, pw = max(0, tile - H0), max(0, tile - W0)
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph))
    _, _, H, W = x.shape
    pos = [(y, xx) for y in tile_positions(H, tile, stride) for xx in tile_positions(W, tile, stride)]
    wwin, wsum, out = _blend(tile), torch.zeros(1, 1, H, W, device=x.device), None
    per = max(1, max_batch // B)
    for i in range(0, len(pos), per):
        chunk = pos[i:i + per]
        ob = fn(torch.cat([x[:, :, y:y + tile, xx:xx + tile] for y, xx in chunk], 0)).float()
        if out is None:
            out = torch.zeros(B, ob.shape[1], H, W, device=x.device)
        for j, (y, xx) in enumerate(chunk):
            out[:, :, y:y + tile, xx:xx + tile] += ob[j * B:(j + 1) * B] * wwin
            wsum[:, :, y:y + tile, xx:xx + tile] += wwin
    return (out / wsum)[:, :, :H0, :W0]

@torch.no_grad()
def ddim_sample(model, cond, extra, n, gen):
    _, C, H, W = cond.shape; T = AB.shape[0]
    ts = np.unique(np.linspace(0, T - 1, CFG["DDIM_STEPS"]).round().astype(int))[::-1]
    x = torch.randn((n, 1, H, W), device=DEVICE, generator=gen); cn = cond.expand(n, -1, -1, -1)
    for i, t in enumerate(ts):
        tp = int(ts[i + 1]) if i + 1 < len(ts) else -1
        a_t = AB[int(t)]; a_p = AB[tp] if tp >= 0 else torch.tensor(1.0, device=DEVICE)
        def fn(xb, t=int(t)):
            with autocast_ctx():
                return model.den(xb, torch.full((xb.shape[0],), t, device=DEVICE, dtype=torch.long),
                                 extra.expand(xb.shape[0], -1))
        v = tiled(fn, torch.cat([x, cn], 1), CFG["PATCH"], CFG["TILE_STRIDE"])
        x0h = (a_t.sqrt() * x - (1 - a_t).sqrt() * v).clamp(-8, 8)
        eh = (1 - a_t).sqrt() * x + a_t.sqrt() * v
        if tp < 0:
            return x0h
        sig = CFG["DDIM_ETA"] * torch.sqrt((1 - a_p) / (1 - a_t)) * torch.sqrt(1 - a_t / a_p)
        x = a_p.sqrt() * x0h + torch.sqrt((1 - a_p - sig ** 2).clamp_min(0)) * eh + \
            sig * torch.randn(x.shape, device=DEVICE, generator=gen)
    return x

def _to_dev(inp):
    return (torch.from_numpy(inp["cond"])[None].to(DEVICE), torch.from_numpy(inp["extra"])[None].to(DEVICE),
            torch.from_numpy(inp["obs"].astype(np.float32))[None, None].to(DEVICE),
            torch.from_numpy(inp["x0"])[None, None].to(DEVICE), torch.from_numpy(inp["pstat"])[None].to(DEVICE))

def make_coda_predictor(model, seed):
    model.eval()
    @torch.no_grad()
    def predict(inp, n, kind, t):
        cond, extra, obs, x0, pstat = _to_dev(inp)
        gen = torch.Generator(device=DEVICE); gen.manual_seed(int(seed * 1_000_003 + t * 7 + KIND_ID[kind]))
        xs = [ddim_sample(model, cond, extra, min(CFG["SAMPLE_CHUNK"], n - k), gen) for k in range(0, n, CFG["SAMPLE_CHUNK"])]
        comp = obs * x0 + (1 - obs) * torch.cat(xs, 0)
        f = pfz_features(comp, obs.expand_as(comp), cond.expand(n, -1, -1, -1), pstat.expand(n, -1, -1, -1), model.nctx)
        def fn(fb):
            with autocast_ctx():
                return model.pfz(fb, None, extra.expand(fb.shape[0], -1))
        p = torch.sigmoid(tiled(fn, f, CFG["PATCH"], CFG["TILE_STRIDE"]))
        return comp[:, 0].cpu().numpy(), p[:, 0].cpu().numpy()
    return predict

def make_gauss_predictor(model, seed):
    model.eval()
    @torch.no_grad()
    def predict(inp, n, kind, t):
        cond, extra, *_ = _to_dev(inp)
        def fn(xb):
            with autocast_ctx():
                return model.net(xb, None, extra.expand(xb.shape[0], -1))
        o = tiled(fn, cond, CFG["PATCH"], CFG["TILE_STRIDE"])[0].cpu().numpy()
        rng = np.random.default_rng([seed, t, KIND_ID[kind]])
        ens = o[0][None] + np.exp(0.5 * np.clip(o[1], -8, 4))[None] * rng.standard_normal((n,) + o[0].shape)
        return ens.astype(np.float32), None
    return predict

def make_envpfz_predictor(model):
    model.eval()
    @torch.no_grad()
    def predict(inp, n, kind, t):
        cond, extra, _, _, pstat = _to_dev(inp)
        def fn(fb):
            with autocast_ctx():
                return model.net(fb, None, extra.expand(fb.shape[0], -1))
        return None, torch.sigmoid(tiled(fn, model.feats(cond, pstat), CFG["PATCH"], CFG["TILE_STRIDE"]))[:, 0].cpu().numpy()
    return predict

In [ ]:
# ============================== STAGE 1: DATA ==============================
if CFG["PROCESSED_IN"]:
    hits = sorted(glob.glob(os.path.join(CFG["PROCESSED_IN"], "**", "logchl.npy"), recursive=True))
    assert hits, f"no logchl.npy under {CFG['PROCESSED_IN']}"
    PROC_SRC = Path(hits[0]).parent; D = load_processed(PROC_SRC); log("loaded processed data from", PROC_SRC)
elif CFG["DATA_MODE"] == "synthetic":
    PROC_SRC = None; log("generating synthetic data ..."); D = make_synthetic(CFG, seed=123)
else:
    PROC_SRC = PROC
    if CFG["STAGES"]["download"]:
        download_chl(CFG); download_cmems(CFG)
        if CFG["GFW_MODE"] == "api":
            download_gfw_api(CFG)
    if CFG["STAGES"]["preprocess"]:
        D = preprocess_real(CFG); save_processed(D, PROC)
    else:
        D = load_processed(PROC)
D = prepare(D, CFG)

In [ ]:
# ============================== STAGE 2: FIXED EVALUATION MASKS (shared by every method) ==============================
BANK = MaskBank(D, CFG)
mask_file = (PROC_SRC / "eval_masks.npz") if PROC_SRC is not None and (PROC_SRC / "eval_masks.npz").exists() else None
if mask_file is not None:
    EVS = load_eval_sets(mask_file); log("loaded fixed eval masks from", mask_file)
else:
    EVS = dict(test=build_eval_set(D, BANK, CFG, "test", CFG["N_EVAL_DAYS"], EVAL_KINDS, CFG["EVAL_SEED"]),
               val=build_eval_set(D, BANK, CFG, "val", CFG["N_VAL_EVAL_DAYS"], ["cloud"], CFG["EVAL_SEED"] + 1))
    save_eval_sets(PROC / "eval_masks.npz", EVS); log("saved eval masks ->", PROC / "eval_masks.npz")
EV_TEST, EV_VAL = EVS["test"], EVS["val"]
_s = make_train_sample(D, BANK, CFG, np.random.default_rng(0))
log("train sample shapes:", {k: v.shape for k, v in _s.items()})

In [ ]:
# ============================== STAGE 3: CHEAP BASELINES (CPU) ==============================
PFZ_LAG = CFG["PFZ_WINDOW"][1] - CFG["PFZ_WINDOW"][0]
def pred_clim_fill(inp, n, kind, t):
    return D["Zclim"][D["month"][t] - 1][None], None
def pred_clim_pfz(inp, n, kind, t):
    return None, D["pclim"][D["month"][t] - 1][None]
def pred_persist_pfz(inp, n, kind, t):
    tp = t - PFZ_LAG      # previous, non-overlapping label window (assumes AIS available with a few days' latency)
    return None, (D["Y"][tp].astype(np.float32) if tp >= 0 else D["pclim"][D["month"][t] - 1])[None]

if CFG["STAGES"]["baselines_cpu"]:
    save_eval(RUNS / "ClimFill", evaluate("ClimFill", pred_clim_fill, D, EV_TEST, CFG, 1))
    for nm, fn in (("ClimPFZ", pred_clim_pfz), ("PersistPFZ", pred_persist_pfz)):
        th = theta_from_val(nm, fn, D, EV_VAL, CFG, 1)
        save_eval(RUNS / nm, evaluate(nm, fn, D, EV_TEST, CFG, 1, theta=th, kinds=["cloud"]), dict(theta=th))
if CFG["STAGES"]["dineof"]:
    DIN = run_dineof(D, EV_TEST, CFG)
    def pred_dineof(inp, n, kind, t):
        f = np.zeros(D["ocean"].shape, np.float32); f[D["ocean"]] = DIN[kind][t]; return f[None], None
    save_eval(RUNS / "DINEOF", evaluate("DINEOF", pred_dineof, D, EV_TEST, CFG, 1))

In [ ]:
# [TORCH] ============================== STAGE 4: TRAIN + EVALUATE CODA (coupled / detached) ==============================
if CFG["STAGES"]["train"]:
    for mode, seed in CFG["RUN_PLAN"]:
        name = f"CODA-{mode}-s{seed}"; rd = RUNS / name
        if (rd / "metrics_rows.csv").exists():
            log(name, "already evaluated - skipping"); continue
        model, hist, done = fit(CODA(CFG), coda_loss_fn(mode), CFG["MAX_STEPS"], rd, name, seed)
        if not done:
            continue
        pred = make_coda_predictor(model, seed)
        th = theta_from_val(name, pred, D, EV_VAL, CFG, min(8, CFG["N_SAMPLES"]))
        res = evaluate(name, pred, D, EV_TEST, CFG, CFG["N_SAMPLES"], CFG["N_SAMPLES_UQ"], CFG["N_UQ_DAYS"], theta=th)
        save_eval(rd, res, dict(mode=mode, seed=seed, theta=th, cfg=CFG))
        del model; torch.cuda.empty_cache()

In [ ]:
# [TORCH] ============================== STAGE 5: NEURAL BASELINES ==============================
if CFG["STAGES"]["baseline_unet"]:
    seed = CFG["RUN_PLAN"][0][1]; name = f"GaussUNet-s{seed}"; rd = RUNS / name
    if not (rd / "metrics_rows.csv").exists():
        net, _, done = fit(GaussUNet(CFG), gauss_loss_fn, CFG["UNET_STEPS"], rd, name, seed)
        if done:
            save_eval(rd, evaluate(name, make_gauss_predictor(net, seed), D, EV_TEST, CFG, CFG["N_SAMPLES"],
                                   CFG["N_SAMPLES_UQ"], CFG["N_UQ_DAYS"]), dict(seed=seed))
if CFG["STAGES"]["baseline_env_pfz"]:
    seed = CFG["RUN_PLAN"][0][1]; name = f"EnvPFZ-s{seed}"; rd = RUNS / name
    if not (rd / "metrics_rows.csv").exists():
        net, _, done = fit(EnvPFZ(CFG), envpfz_loss_fn, CFG["ENV_PFZ_STEPS"], rd, name, seed)
        if done:
            pred = make_envpfz_predictor(net); th = theta_from_val(name, pred, D, EV_VAL, CFG, 1)
            save_eval(rd, evaluate(name, pred, D, EV_TEST, CFG, 1, theta=th, kinds=["cloud"]), dict(seed=seed, theta=th))

In [ ]:
# ============================== STAGE 6: COMPARISON TABLES + PAIRED STATISTICS ==============================
def load_runs():
    dirs = {}
    for p in sorted(glob.glob(CFG["EXTRA_RUN_GLOB"], recursive=True)) + [str(d) for d in RUNS.glob("*")]:
        p = Path(p)
        if (p / "metrics_rows.csv").exists():
            dirs[p.name] = p                    # later entries (this session) win
    rows = pd.concat([pd.read_csv(d / "metrics_rows.csv") for d in dirs.values()], ignore_index=True) if dirs else pd.DataFrame()
    pfz = {n: json.loads((d / "pfz_summary.json").read_text()) for n, d in dirs.items() if (d / "pfz_summary.json").exists()}
    return dirs, rows, pfz

def paired_compare(df, metric, A, B, kind, block=3, n_boot=5000, seed=0):
    from scipy.stats import wilcoxon
    a = df[(df.model == A) & (df.kind == kind)].set_index("day")[metric]
    b = df[(df.model == B) & (df.kind == kind)].set_index("day")[metric]
    idx = a.index.intersection(b.index).sort_values(); d = (a[idx] - b[idx]).dropna().values
    if len(d) < 5:
        return dict(n=len(d))
    rng = np.random.default_rng(seed); nb = int(np.ceil(len(d) / block)); boots = []
    for _ in range(n_boot):       # moving-block bootstrap: eval days are autocorrelated
        st = rng.integers(0, len(d) - block + 1, nb); boots.append(np.concatenate([d[s:s + block] for s in st])[:len(d)].mean())
    p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return dict(n=len(d), mean_diff=float(d.mean()), ci95=(float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))),
                p_wilcoxon=float(p))

RUN_DIRS, ROWS, PFZS = {}, pd.DataFrame(), {}
if CFG["STAGES"]["compare"]:
    RUN_DIRS, ROWS, PFZS = load_runs()
    if len(ROWS):
        ROWS["base"] = ROWS["model"].str.replace(r"-s\d+$", "", regex=True)
        mets = [m for m in ["rmse", "mae", "bias", "crps", "ssr", "cov90", "ssim", "grad_mae_mean", "front_f1_mean",
                            "front_f1_s0"] if m in ROWS]
        rec = ROWS.dropna(subset=["rmse"]) if "rmse" in ROWS else ROWS.iloc[:0]
        if len(rec):
            per_seed = rec.groupby(["base", "model", "kind"])[mets].mean()
            tab = per_seed.groupby(["base", "kind"]).agg(["mean", "std"]).round(4)
            print("\n=== Reconstruction (log10 Chl, hidden pixels; mean over eval days, then mean/std over seeds) ===")
            print(tab.to_string()); tab.to_csv(WORK / "table_reconstruction.csv")
        prow = [dict(model=n, kind=k, **v) for n, dct in PFZS.items() for k, v in dct.items()]
        if prow:
            ptab = pd.DataFrame(prow).round(4)
            print("\n=== PFZ (pooled footprint pixels; BSS is skill vs monthly climatology; theta chosen on val) ===")
            print(ptab.to_string(index=False)); ptab.to_csv(WORK / "table_pfz.csv", index=False)
        tests = []
        for s in sorted({int(m.split("-s")[-1]) for m in ROWS.model.unique() if m.startswith("CODA-")}):
            A, B = f"CODA-coupled-s{s}", f"CODA-detached-s{s}"
            if {A, B} <= set(ROWS.model):
                for kind in EVAL_KINDS:
                    for met in ["rmse", "crps", "front_f1_s0", "grad_mae_s0", "pfz_brier"]:
                        if met in ROWS:
                            tests.append(dict(seed=s, kind=kind, metric=met, **paired_compare(ROWS, met, A, B, kind)))
        if tests:
            tt = pd.DataFrame(tests)
            print("\n=== Coupled minus uncoupled (negative = coupled lower). Per seed, paired over identical days+masks ===")
            print(tt.to_string(index=False)); tt.to_csv(WORK / "table_coupling_tests.csv", index=False)
            print("Reminder: few eval days -> low power; do not call a difference 'significant' from one seed.")

In [ ]:
# ============================== STAGE 7: FIGURES ==============================
import matplotlib
import matplotlib.pyplot as plt

def _arr(run, key):
    f = RUN_DIRS[run] / "eval_arrays.npz"
    if not f.exists():
        return None
    z = np.load(f); return z[key].astype(np.float32) if key in z.files else None

if CFG["STAGES"]["compare"] and len(ROWS):
    cm = matplotlib.colormaps["viridis"].copy(); cm.set_bad("0.85")
    # --- Fig A: qualitative panel (cloud mask, first kept day) ---
    kind = "cloud"; t = int(EV_TEST["days"][0]); mu_, sd_ = D["stats"]["logchl"]
    inp = build_inputs(D, t, list(EV_TEST["masks"][kind][0]), None, CFG)
    panels = [("truth", np.where(D["V"][t], D["logchl"][t], np.nan)),
              ("input", np.where(inp["obs"], D["logchl"][t], np.nan))]
    for run in [r for r in ["DINEOF", "ClimFill"] + sorted(RUN_DIRS) if r in RUN_DIRS]:
        m = _arr(run, f"{kind}__map0__s0")
        if m is not None and run not in [p[0] for p in panels]:
            panels.append((run, np.where(D["ocean"], m, np.nan)))
    n = len(panels); fig, ax = plt.subplots(1, n, figsize=(3 * n, 3.4))
    vmin, vmax = np.nanpercentile(panels[0][1], [2, 98])
    for a, (ttl, f) in zip(np.atleast_1d(ax), panels):
        a.imshow(f, origin="lower", cmap=cm, vmin=vmin, vmax=vmax); a.set_title(ttl, fontsize=8); a.axis("off")
    fig.suptitle(f"log10 Chl, {str(D['dates'][t])}, cloud-transfer mask (models: sample 0)"); fig.tight_layout()
    fig.savefig(FIGS / "fig_panel.png", dpi=150); plt.show()
    # --- Fig B: PFZ maps + uncertainty for CODA runs ---
    cr = [r for r in sorted(RUN_DIRS) if r.startswith("CODA-")]
    if cr:
        fig, ax = plt.subplots(len(cr), 3, figsize=(9, 3 * len(cr)), squeeze=False)
        for i, r in enumerate(cr):
            for j, (key, ttl) in enumerate([("pfz", "P(PFZ)"), ("pfzstd", "sd P(PFZ)"), ("std", "sd log10 Chl")]):
                m = _arr(r, f"{kind}__map0__{key}")
                if m is not None:
                    im = ax[i, j].imshow(np.where(D["ocean"], m, np.nan), origin="lower", cmap=cm); plt.colorbar(im, ax=ax[i, j], fraction=0.046)
                ax[i, j].set_title(f"{r}: {ttl}", fontsize=8); ax[i, j].axis("off")
        fig.tight_layout(); fig.savefig(FIGS / "fig_pfz_uq.png", dpi=150); plt.show()
    # --- Fig C: reliability + Fig D: error vs predicted spread + Fig E: PSD ---
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    for r in sorted(RUN_DIRS):
        p, y = _arr(r, "cloud__pfz_p"), _arr(r, "cloud__pfz_y")
        if p is not None:
            rel = reliability(p, y); ax[0].plot(rel[:, 0], rel[:, 1], "o-", label=r, ms=3)
        e, s = _arr(r, "cloud__uq_err"), _arr(r, "cloud__uq_std")
        if e is not None and len(e) > 20:
            q = np.quantile(s, np.linspace(0, 1, 11)); b = np.clip(np.digitize(s, q[1:-1]), 0, 9)
            ax[1].plot([s[b == k].mean() for k in range(10)], [np.sqrt(np.mean(e[b == k] ** 2)) for k in range(10)], "o-", label=r, ms=3)
        pt, pm = _arr(r, "cloud__psd_true"), _arr(r, "cloud__psd_s0")
        if pt is not None:
            k = np.arange(1, len(pt) + 1)
            if not ax[2].lines:
                ax[2].plot(k, pt, "k-", lw=2, label="truth")
            ax[2].plot(k, pm, label=r)
    ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set(xlabel="predicted P(PFZ)", ylabel="observed frequency", title="Reliability (cloud masks)")
    lim = ax[1].get_xlim(); ax[1].plot(lim, lim, "k--", lw=0.8); ax[1].set(xlabel="predicted sd", ylabel="RMSE", title="Error vs spread (binned)")
    ax[2].set(xscale="log", xlabel="wavenumber (cycles / window)", ylabel="log10 PSD", title="Spectra (sample 0)")
    for a in ax:
        if a.get_legend_handles_labels()[0]:
            a.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIGS / "fig_calib_uq_psd.png", dpi=150); plt.show()
    # --- Fig F: training curves ---
    hs = {r: pd.read_csv(RUN_DIRS[r] / "train_hist.csv") for r in cr if (RUN_DIRS[r] / "train_hist.csv").exists()}
    if hs:
        cols = [c for c in ["diff", "recon", "grad", "pfz", "adv"] if any(c in h for h in hs.values())]
        fig, ax = plt.subplots(1, len(cols), figsize=(3.5 * len(cols), 3), squeeze=False)
        for j, c in enumerate(cols):
            for r, h in hs.items():
                if c in h:
                    ax[0, j].plot(h["step"], h[c], label=r)
            ax[0, j].set_title(f"weighted loss: {c}", fontsize=9); ax[0, j].set_yscale("log")
        ax[0, 0].legend(fontsize=7); fig.tight_layout(); fig.savefig(FIGS / "fig_train.png", dpi=150); plt.show()
    log("figures ->", FIGS)

## Reading the outputs honestly
- **Synthetic mode results mean nothing scientifically.** They only prove the code runs.
- The central claim needs `CODA-coupled` vs `CODA-detached` over **several seeds** (≥3 if the GPU budget allows), on the same days and masks. Look at the paired table: a CI that straddles zero means "no detectable effect", which is a legitimate finding and must be reported as such.
- PFZ skill must be read against `ClimPFZ` and `PersistPFZ` (BSS column). If CODA does not beat climatology, the Chl reconstruction is not adding operational value for this proxy.
- `front_f1_s0` vs `front_f1_mean`: ensemble means smooth fronts by construction; individual samples are what should preserve structure.
- `ssr` ≈ 1 and `cov90` ≈ 0.9 indicate a well-dispersed ensemble; `ssr` < 1 means over-confident.

## Known limitations of v0.1 (fix or disclose before submission)
- DINEOF sees the whole test time series (transductive); CODA sees only t±1. OI-SwinUNet, Barth et al. diffusion and PPIDM are **not** implemented here — run their official code on the same `eval_masks.npz` days.
- The advection term uses geostrophic currents only (no Ekman, no biology); it is a weak prior, not physics. Ablate it (`LAMBDA["adv"]=0`).
- The GFW proxy covers AIS-equipped vessels only; coverage in parts of the SCS may be incomplete. Consider a second proxy (e.g. night-time light boat detections) as a robustness check.
- The PFZ label window includes days t+1..t+4, i.e. this is a nowcast-style label, not a forecast.